# Colab T4 GPU compatibility check

In Colab, choose **Runtime → Change runtime type → T4 GPU**, then run the cells in order. This notebook checks the selected runtime only: GPU name and VRAM, PyTorch/CUDA visibility, a tiny tensor operation, memory use, and a JSON diagnostic report. It does not install packages, download HeyGem models, start Docker, or run avatar/video generation. A Colab GPU is not connected to your local VS Code or Docker services.

In [1]:
import json
import platform
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import torch

report = {
    'checked_at_utc': datetime.now(timezone.utc).isoformat(),
    'platform': platform.platform(),
    'python_version': platform.python_version(),
    'pytorch_version': torch.__version__,
    'pytorch_cuda_build': torch.version.cuda,
    'cuda_available': torch.cuda.is_available(),
    'gpu_devices': [],
    'tensor_test': None,
}

print('PyTorch:', report['pytorch_version'])
print('PyTorch CUDA build:', report['pytorch_cuda_build'])
print('CUDA available:', report['cuda_available'])

if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        device = {
            'index': index,
            'name': props.name,
            'vram_bytes': props.total_memory,
            'vram_gib': round(props.total_memory / 1024**3, 2),
            'compute_capability': list(torch.cuda.get_device_capability(index)),
        }
        report['gpu_devices'].append(device)
        print(f"GPU {index}: {device['name']} | {device['vram_gib']} GiB VRAM | capability {device['compute_capability']}")
        if 'T4' not in device['name']:
            print('Note: this runtime is not reporting a T4 GPU.')

    try:
        torch.cuda.reset_peak_memory_stats()
        test = torch.arange(1024, dtype=torch.float32, device='cuda')
        result = (test * 2).sum().item()
        report['tensor_test'] = {
            'passed': True,
            'device': str(test.device),
            'result': result,
            'allocated_bytes': torch.cuda.memory_allocated(),
            'reserved_bytes': torch.cuda.memory_reserved(),
            'peak_allocated_bytes': torch.cuda.max_memory_allocated(),
        }
        print('Small CUDA tensor test: PASS; result =', result)
        print('Memory (allocated / reserved / peak bytes):', report['tensor_test']['allocated_bytes'], '/', report['tensor_test']['reserved_bytes'], '/', report['tensor_test']['peak_allocated_bytes'])
        del test
    except RuntimeError as error:
        report['tensor_test'] = {'passed': False, 'error': str(error)}
        print('Small CUDA tensor test: FAIL;', error)
else:
    print('No CUDA GPU is available. In Colab, select Runtime → Change runtime type → T4 GPU and rerun this cell.')

if shutil.which('nvidia-smi'):
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.used,driver_version', '--format=csv,noheader,nounits'],
        capture_output=True, text=True, check=False, timeout=15,
    )
    report['nvidia_smi_returncode'] = result.returncode
    report['nvidia_smi'] = result.stdout.strip() if result.returncode == 0 else result.stderr.strip()
    print('\nnvidia-smi (name, total MiB, used MiB, driver):\n' + report['nvidia_smi'])
else:
    report['nvidia_smi'] = 'nvidia-smi is not available in this runtime.'
    print(report['nvidia_smi'])

report_path = Path('/content/heygem_colab_gpu_report.json')
report_path.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
print('\nDiagnostic report saved to:', report_path)

PyTorch: 2.11.0+cu130
PyTorch CUDA build: 13.0
CUDA available: True
GPU 0: Tesla T4 | 14.56 GiB VRAM | capability [7, 5]
Small CUDA tensor test: PASS; result = 1047552.0
Memory (allocated / reserved / peak bytes): 4096 / 2097152 / 8704

nvidia-smi (name, total MiB, used MiB, driver):
Tesla T4, 15360, 123, 580.82.07

Diagnostic report saved to: /content/heygem_colab_gpu_report.json


In [2]:
# Optional: download the small JSON diagnostic to your computer.
from google.colab import files
files.download('/content/heygem_colab_gpu_report.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Interpreting the result

A passing tensor test shows only that this notebook can use the selected CUDA device. It does not establish that HeyGem's Docker images or models fit the GPU or work in Colab. HeyGem's documented service deployment needs NVIDIA-enabled Docker Compose and substantial RAM/storage; use a dedicated, supported GPU host for that deployment and follow current project instructions.